# QLoRA: Quantized Low-Rank Adaptation

QLoRA (Dettmers et al. 2023) combined 4-bit quantization with LoRA to enable fine-tuning of 65B-parameter models on a single 48GB GPU. It unlocked fine-tuning large models for academic researchers and practitioners without access to multi-GPU clusters.

## The Quantization-LoRA Combination

Quantization reduces model precision from 32-bit or 16-bit floats to lower-bit integer representations. 4-bit quantization reduces memory by 8x compared to fp32, but naive quantization degrades quality significantly.

QLoRA's key innovations:

**NF4 (Normal Float 4)**: a 4-bit data type optimized for normally-distributed weight values. Uses non-uniform quantization buckets that allocate more precision near zero (where most weights cluster) and less precision in the tails. Significantly better quality than uniform 4-bit.

**Double quantization**: quantize the quantization constants themselves, saving an additional ~0.37 bits per parameter.

**Paged optimizers**: use NVIDIA's unified memory to page optimizer states between GPU and CPU RAM, preventing OOM during occasional gradient spikes.

The base model is frozen in 4-bit; LoRA adapters are trained in bf16. Gradients flow through the quantized weights via straight-through estimators.

In [ ]:
# Demonstrate NF4 quantization scheme vs uniform quantization
import math

def uniform_quantize(value: float, n_bits: int = 4, vmin: float = -1.0, vmax: float = 1.0) -> float:
    n_levels = 2 ** n_bits
    step = (vmax - vmin) / (n_levels - 1)
    quantized_idx = round((value - vmin) / step)
    quantized_idx = max(0, min(n_levels - 1, quantized_idx))
    return vmin + quantized_idx * step

def nf4_quantize(value: float) -> float:
    # NF4 quantization levels (from Dettmers et al.)
    # Non-uniform levels optimized for normal distribution
    nf4_levels = [
        -1.0, -0.6962, -0.5251, -0.3949,
        -0.2844, -0.1848, -0.0911, 0.0,
         0.0796,  0.1609,  0.2461,  0.3379,
         0.4407,  0.5626,  0.7230,  1.0
    ]
    # Find nearest level
    return min(nf4_levels, key=lambda x: abs(x - value))

# Test on values drawn from N(0, 0.1) -- typical weight distribution
import random
rng = random.Random(42)
weights = [rng.gauss(0, 0.1) for _ in range(1000)]
weights_clipped = [max(-1, min(1, w)) for w in weights]

unif_errors = [abs(w - uniform_quantize(w)) for w in weights_clipped]
nf4_errors  = [abs(w - nf4_quantize(w))    for w in weights_clipped]

print('Quantization error comparison (4-bit, N(0,0.1) weights):')
print(f'  Uniform 4-bit - Mean abs error: {sum(unif_errors)/len(unif_errors):.6f}')
print(f'  NF4           - Mean abs error: {sum(nf4_errors)/len(nf4_errors):.6f}')
print(f'  NF4 improvement: {(sum(unif_errors)-sum(nf4_errors))/sum(unif_errors):.1%}')

## Memory Math: QLoRA vs LoRA vs Full Fine-Tune

For a 7B parameter model:

| Method | Base Model | Adapters | Optimizer | Total |
|--------|-----------|----------|-----------|-------|
| Full FT (bf16+fp32) | 14GB | — | 56GB | ~112GB |
| LoRA (bf16) | 14GB | 0.3GB | 1.2GB | ~16GB |
| QLoRA (4-bit) | 3.5GB | 0.3GB | 1.2GB | ~6GB |

QLoRA makes 7B fine-tuning feasible on a single 16GB GPU (RTX 4080), and 13B feasible on a 24GB GPU (RTX 3090/4090). This democratized fine-tuning.

In [ ]:
def qlora_memory_estimate(n_params_billions: float, rank: int = 16,
                           lora_target_pct: float = 0.01) -> dict:
    n = n_params_billions * 1e9
    lora_params = n * lora_target_pct
    # 4-bit base model: 0.5 bytes/param
    base_mem_gb = n * 0.5 / 1e9
    # LoRA in bf16: 2 bytes/param
    adapter_mem_gb = lora_params * 2 / 1e9
    # Adam optimizer in fp32: 2 * 4 bytes/param
    optimizer_mem_gb = lora_params * 8 / 1e9
    # Activation memory (rough estimate)
    activation_mem_gb = 1.0  # varies with batch size
    total = base_mem_gb + adapter_mem_gb + optimizer_mem_gb + activation_mem_gb
    return {
        'model_size': f'{n_params_billions}B',
        'base_4bit_GB': round(base_mem_gb, 1),
        'adapters_GB': round(adapter_mem_gb, 2),
        'optimizer_GB': round(optimizer_mem_gb, 2),
        'activations_GB': activation_mem_gb,
        'total_GB': round(total, 1),
        'fits_on': (
            '8GB GPU' if total <= 8 else
            '16GB GPU' if total <= 16 else
            '24GB GPU' if total <= 24 else
            '40GB GPU' if total <= 40 else
            '80GB GPU' if total <= 80 else 'multi-GPU'
        )
    }

print(f'{'Size':<10} {'Base':>8} {'Adapters':>9} {'Optimizer':>10} {'Total':>8} {'Fits'}')
print('-' * 60)
for b in [3, 7, 13, 34, 70]:
    r = qlora_memory_estimate(b)
    print(f"{r['model_size']:<10} {r['base_4bit_GB']:>7}GB {r['adapters_GB']:>8}GB {r['optimizer_GB']:>9}GB {r['total_GB']:>7}GB  {r['fits_on']}")

## Practical QLoRA Training Setup

Key hyperparameters for QLoRA:
- **Bits**: 4 (NF4) for maximum memory efficiency; 8-bit for slightly better quality at 2x memory
- **Rank**: 16-64; start with 64 for instruction tuning since you have headroom
- **Alpha**: set equal to rank (scaling=1.0) or 2x rank for stronger adaptation
- **Target modules**: all linear layers (`q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj`) for best results
- **Learning rate**: 2e-4 is a reliable starting point
- **Batch size**: 4-8 with gradient accumulation 4-8 steps; effective batch 16-64
- **Epochs**: 1-3 for instruction tuning; overfitting happens quickly on small datasets

## QLoRA Quality vs Full Fine-Tune

The main concern with QLoRA is quantization-induced quality degradation. In practice:
- QLoRA fine-tuned Llama-2 13B outperforms full fine-tuned Llama-2 7B on most tasks
- The quantization noise acts as slight regularization, sometimes reducing overfitting on small datasets
- For tasks requiring high numerical precision (math, code), the 4-bit quantization can hurt more
- bf16 fine-tuning (16-bit, full LoRA) and QLoRA produce similar results when the dataset is >10K examples

The QLoRA paper showed Guanaco (QLoRA fine-tuned LLaMA-65B) reached near-ChatGPT performance on human evaluation benchmarks — establishing that quantization did not meaningfully degrade fine-tune quality.